In [ ]:
!pip install gdown

In [ ]:
import gdown
gdown.download('https://bit.ly/3q9SZix', '20s_best_book.json', quiet=False)

In [ ]:
import pandas as pd
books_df = pd.read_json('20s_best_book.json')
books_df.columns

In [ ]:
books = books_df[['no','ranking','bookname','authors','publisher','publication_year','isbn13']]
books.head()

- loc 메서드: 판다스가 제공하는 데이터프레임 객체 행과 열 선택하기  
- 대괄호를 사용하여 행의 목록과 열의 목록을 받음  

```python
books_df.loc[[0,1], ['bookname','authors']] # 0,1 행, bookname, authors 열 선택
```  
- loc 메서드 vs iloc 메서드

| 구분 | loc | iloc |
| :--- | :--- | :--- |
| **기존 기준** | 행/열의 **라벨(이름)** 기준 | 행/열의 **정수 위치(Index 번호)** 기준 |
| **슬라이싱 범위** | `[a:b]` 사용 시 **`b` 포함** | `[a:b]` 사용 시 **`b` 미포함** |
| **조건문 검색** | **가능** (예: `df.loc[df['Age'] > 20]`) | **불가** |
| **열 지정 방식** | 열 이름을 문자열로 지정 (`'Age'`, `'Name'`) | 열의 위치 번호로 지정 (`0`, `1`) |

In [ ]:
# loc 메서드 - 슬라이싱 사용 예시
# books = books_df.loc[:, 'no':'bookname']
# books.head()

In [ ]:
# step 지정 가능
# books_df.loc[::2, 'no':'isbn13'].head()

In [ ]:
# requests.get() 함수 사용
import requests
isbn = 9791190090018 
url = 'https://www.yes24.com/Product/Search?domain=BOOK&query={}'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36'}

res = requests.get(url, headers=headers)
r = requests.get(url.format(isbn))  # format 함수 사용
print(r.text)

- BeautifulSoup: 파이썬에서 웹 스크래핑을 할 때 HTML에서 원하는 내용을 쉽게 추출할 수 있도록 도와주는 라이브러리

In [64]:
from bs4 import BeautifulSoup 

- beautifulsoup를 사용하려면
    1. 클래스 객체 생성
    2. 첫 번째 매개변수는 파싱할 html 문서
    3. 두 번째 매개변수는 파싱에 사용할 파서 

In [40]:
# beautifulsoup는 2가지 파서 존재
# 1) 파이썬에 기본 내장된 html 파서
# 2) 외부 lxml 패키지

soup = BeautifulSoup(r.text, 'html.parser')

- find() 메서드: HTML 태그명과 속성을 매개변수로 직접 지정하여 조건에 맞는 첫 번째 요소를 찾는 방식  
    - 첫 번째 매개변수는 찾을 태그 이름 지정  
    - attrs 매개변수에는 찾으려는 태그의 속성을 딕셔너리로 지정  

- select() 메서드: CSS 선택자 문법(`.`, `#`, `>`)을 활용하여 조건에 맞는 모든 요소를 리스트로 찾는 방식

In [46]:
# find() 사용, Tag 형태로 반환
prd_link = soup.find('a', attrs={'class':'gd_name'})
print(type(prd_link))

<class 'bs4.element.Tag'>


In [47]:
print(prd_link['href'])

/product/goods/74261416


In [ ]:
# select() 사용, ResultSet 형태로 반환
# prd_link = soup.select("a.gd_name")
# print(type(prd_link))

In [ ]:
# 상세 페이지 가져오기
url = 'http://www.yes24.com' + prd_link['href']
r = requests.get(url)
print(r.text)

In [ ]:
soup = BeautifulSoup(r.text, 'html.parser')
prd_detail = soup.find('div', attrs={'id':'infoset_specific'})
print(prd_detail)

- find_all() 메서드: HTML 태그명과 속성을 매개변수로 직접 지정하여 조건에 맞는 모든 요소를 리스트로 찾는 방식
    - 첫 번째 매개변수는 찾을 태그 이름 지정
    - attrs 매개변수에는 찾으려는 태그의 속성을 딕셔너리로 지정

- select_one() 메서드: CSS 선택자 문법(`.`, `#`, `>`)을 활용하여 조건에 맞는 첫 번째 요소를 찾는 방식
    - 매개변수로 찾으려는 요소의 CSS 선택자 문자열 지정

In [53]:
# find_all() 사용
prd_tr_list = prd_detail.find_all('tr')
print(type(prd_tr_list))

<class 'bs4.element.ResultSet'>


In [ ]:
# select_one() 사용
# prd_tr_list = prd_detail.select_one('tr')
# print(type(prd_tr_list))

<class 'bs4.element.Tag'>


- `.text` (속성): 태그 내부의 모든 텍스트를 가공 없이 있는 그대로 문자열로 가져오는 속성
    - 파이썬 기본 문자열 메서드와 결합하여 `.text.strip()` 형태로 사용 가능

- `get_text()` (메서드): 세부 옵션(매개변수)을 적용하여 텍스트를 가공 및 정리해서 가져올 수 있는 메서드
    - `strip=True`: 하위 태그 각각의 앞뒤 공백을 제거하고 합침

In [ ]:
for tr in prd_tr_list:
    if tr.find('th').get_text() == '쪽수, 무게, 크기':  # .text.strip()도 가능
        page_td = tr.find('td').get_text()
        break

print(page_td.split()[0])

344쪽


In [73]:
# 하나의 함수로 정의
def get_page_cnt(isbn):
    url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'
    res = requests.get(url.format(isbn))
    soup = BeautifulSoup(res.text, 'html.parser')
    prd_info = soup.find('a', attrs={'class':'gd_name'})
    if prd_info == None:
        return ''

    url = 'http://www.yes24.com' + prd_info['href']
    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')
    prd_detail = soup.find('div', attrs={'id':'infoset_specific'})
    prd_tr_list = prd_detail.find_all('tr')

    for tr in prd_tr_list:
        if tr.find('th').get_text() == '쪽수, 무게, 크기':
            return tr.find('td').get_text().split()[0]

    return ''

get_page_cnt(9791190090018)

'344쪽'

In [ ]:
top10_books = books.head(10)
print(top10_books)

In [75]:
def get_page_cnt2(row):
    isbn = row['isbn13']
    return get_page_cnt(isbn)

- apply() 메서드: 데이터프레임의 행 또는 열에 함수를 일괄 적용시키는 메서드
    - axis 매개변수가 1이면 행에, 0(default)이면 열에 적용

In [ ]:
page_count = top10_books.apply(get_page_cnt2, axis=1)
print(page_count)

In [ ]:
# page_count series 객체에 이름 지정
page_count.name = 'page_count'

- merge() 메서드: 판다스에서 두 데이터프레임을 합치거나, 데이터프레임과 시리즈를 합칠 때 사용
    - 첫 번째, 두 번째 매개변수는 합칠 데이터프레임이나 시리즈 객체
    - 두 객체의 인덱스를 기준으로 합칠 경우에는 `left_index`, `right_index` 매개변수를 `True`로 지정

In [78]:
top10_with_page_count = pd.merge(top10_books, page_count, left_index=True, right_index=True)
print(top10_with_page_count)

   no  ranking                     bookname                  authors  \
0   1        1  우리가 빛의 속도로 갈 수 없다면 :김초엽 소설                  지은이: 김초엽   
1   2        2         달러구트 꿈 백화점.이미예 장편소설                  지은이: 이미예   
2   3        3          지구에서 한아뿐 :정세랑 장편소설                  지은이: 정세랑   
3   4        4           시선으로부터, :정세랑 장편소설                  지은이: 정세랑   
4   5        5               아몬드 :손원평 장편소설                  지은이: 손원평   
5   6        6            피프티 피플 :정세랑 장편소설                  지은이: 정세랑   
6   7        7          목소리를 드릴게요 :정세랑 소설집                  지은이: 정세랑   
7   8        8  나미야 잡화점의 기적 :히가시노 게이고 장편소설   지은이: 히가시노 게이고 ;옮긴이: 양윤옥   
8   9        9                   선량한 차별주의자                    김지혜 지음   
9  10        9              쇼코의 미소 :최은영 소설                  지은이: 최은영   

  publisher publication_year         isbn13 page_count  
0        허블             2019  9791190090018       344쪽  
1     팩토리나인             2020  9791165341909       300쪽  
2        난다             2019  979118